# 드론 술래잡기 RL — Colab 실행판

터미널 없이 그냥 **각 칸 왼쪽의 ▶ 버튼**을 위에서부터 순서대로 누르면 돼.
1번부터 5번까지 순서 지켜서 누르기만 하면 학습 → 결과 확인까지 끝남.


## 1. 라이브러리 설치 (처음 한 번만, ▶ 누르고 1~2분 대기)

In [ ]:
!pip install -q gymnasium stable-baselines3 numpy matplotlib
!pip install -q --upgrade git+https://github.com/utiasDSL/gym-pybullet-drones.git
print('설치 완료')


## 2. 환경 코드 저장 (실제 물리 버전 — gym-pybullet-drones)
4개 로터 추력/토크를 PyBullet이 실제로 시뮬레이션함. ▶ 누르면 `drone_tag_env.py` 생성됨.

In [ ]:
%%writefile drone_tag_env.py
"""
drone_tag_env_pybullet.py

FPV 드론 술래잡기 - 실제 물리 버전 (gym-pybullet-drones 기반)
- Crazyflie 2.X 쿼드콥터 모델(cf2x) 사용, 4개 로터 각각의 추력/토크를
  PyBullet 강체 시뮬레이션으로 계산 (뉴턴-오일러 방정식, 공기저항,
  지면효과(ground effect) 등 실제 물리 반영)
- 드론 0 = 술래(chaser, RL 에이전트가 제어)
- 드론 1 = 도망자(evader, 규칙기반)
- 액션 타입: VEL (목표 방향 + 속력 -> 내부 PID 컨트롤러가 4개 모터 RPM으로 변환)
  -> RL은 "어느 방향으로 얼마나 빠르게"만 출력하면 되고, 그걸 실제 로터 RPM으로
     바꾸는 저수준 제어는 라이브러리가 검증된 PID로 처리함 (실감나는 비행 특성 유지)

의존성: gym-pybullet-drones (pip install --upgrade git+https://github.com/utiasDSL/gym-pybullet-drones.git)
"""

import numpy as np
from gymnasium import spaces

from gym_pybullet_drones.envs.BaseRLAviary import BaseRLAviary
from gym_pybullet_drones.utils.enums import DroneModel, Physics, ActionType, ObservationType


class DroneTagPyBulletEnv(BaseRLAviary):
    """
    관측값 (12차원, 기존 point-mass 버전과 동일한 형식 유지):
        [chaser_pos(3), chaser_vel(3), rel_pos_to_evader(3), rel_vel_to_evader(3)]

    액션 (4차원): VEL 타입 - [dir_x, dir_y, dir_z, speed_fraction], 각 [-1, 1]
    """

    def __init__(
        self,
        arena_xy: float = 7.5,       # 경기장 x,y 반경 (총 15m x 15m) - 실제 물리라 30m는 너무 넓어서 축소 추천
        arena_z: float = 3.0,        # 경기장 최대 고도
        catch_radius: float = 0.3,
        episode_len_sec: int = 20,   # 제한시간(초)
        evader_speed_frac: float = 0.55,  # 도망자 속력 비율 (술래보다 살짝 느리게)
        gui: bool = False,           # Colab에서는 항상 False (헤드리스)
        seed: int | None = None,
    ):
        self.arena_xy = arena_xy
        self.arena_z = arena_z
        self.catch_radius = catch_radius
        self.evader_speed_frac = evader_speed_frac
        self._np_rng = np.random.default_rng(seed)

        init_xyzs = np.array(
            [
                [arena_xy * 0.4, 0.0, 1.0],   # 술래 시작 위치
                [-arena_xy * 0.4, 0.0, 1.0],  # 도망자 시작 위치
            ]
        )

        super().__init__(
            drone_model=DroneModel.CF2X,
            num_drones=2,
            initial_xyzs=init_xyzs,
            physics=Physics.PYB,
            pyb_freq=240,
            ctrl_freq=30,
            gui=gui,
            obs=ObservationType.KIN,
            act=ActionType.VEL,
        )
        self.EPISODE_LEN_SEC = episode_len_sec

        # BaseRLAviary는 기본적으로 드론 2대 모두 RL 액션을 받는 구조인데,
        # 여기선 술래(드론0)만 RL로 제어하고 도망자(드론1)는 규칙기반이라
        # 외부에 노출되는 action_space는 술래 1대분(4차원)만 보이게 덮어씀
        self.action_space = spaces.Box(low=-1.0, high=1.0, shape=(4,), dtype=np.float32)

        high = np.array([np.inf] * 12, dtype=np.float32)
        self.observation_space = spaces.Box(low=-high, high=high, dtype=np.float32)

        self.chaser_pos = init_xyzs[0].copy()
        self.evader_pos = init_xyzs[1].copy()
        self.max_steps = int(episode_len_sec * self.CTRL_FREQ)
        self.step_count = 0

    # ------------------------------------------------------------------ #
    # 도망자 규칙기반 정책 (VEL 액션 포맷으로 반환)
    # ------------------------------------------------------------------ #
    def _evader_action(self):
        chaser_state = self._getDroneStateVector(0)
        evader_state = self._getDroneStateVector(1)
        chaser_pos = chaser_state[0:3]
        evader_pos = evader_state[0:3]

        away = evader_pos - chaser_pos
        dist = np.linalg.norm(away) + 1e-6
        away_dir = away / dist

        to_center = np.array(
            [-evader_pos[0], -evader_pos[1], (self.arena_z / 2) - evader_pos[2]]
        )
        to_center = to_center / (np.linalg.norm(to_center) + 1e-6)

        edge_dist = min(
            self.arena_xy - abs(evader_pos[0]),
            self.arena_xy - abs(evader_pos[1]),
            evader_pos[2],
            self.arena_z - evader_pos[2],
        )
        edge_urgency = np.clip(1.0 - edge_dist / 0.8, 0.0, 1.0)

        jitter = self._np_rng.normal(0, 0.2, size=3)
        direction = (1 - edge_urgency) * away_dir + edge_urgency * to_center + jitter
        direction = direction / (np.linalg.norm(direction) + 1e-6)

        return np.array(
            [direction[0], direction[1], direction[2], self.evader_speed_frac],
            dtype=np.float32,
        )

    def _out_of_bounds(self, pos):
        return (
            abs(pos[0]) > self.arena_xy
            or abs(pos[1]) > self.arena_xy
            or pos[2] < 0.05
            or pos[2] > self.arena_z
        )

    # ------------------------------------------------------------------ #
    # Gymnasium 오버라이드
    # ------------------------------------------------------------------ #
    def reset(self, *, seed=None, options=None):
        if seed is not None:
            self._np_rng = np.random.default_rng(seed)

        # 매 에피소드 시작 위치를 랜덤화 (경기장 안쪽, 너무 가깝지 않게)
        while True:
            c_xy = self._np_rng.uniform(-self.arena_xy * 0.7, self.arena_xy * 0.7, size=2)
            e_xy = self._np_rng.uniform(-self.arena_xy * 0.7, self.arena_xy * 0.7, size=2)
            c_pos = np.array([c_xy[0], c_xy[1], self._np_rng.uniform(0.8, self.arena_z * 0.7)])
            e_pos = np.array([e_xy[0], e_xy[1], self._np_rng.uniform(0.8, self.arena_z * 0.7)])
            if np.linalg.norm(c_pos - e_pos) > 1.5:
                break
        self.INIT_XYZS = np.array([c_pos, e_pos])

        obs, info = super().reset(seed=seed, options=options)
        self.step_count = 0
        self.chaser_pos = self._getDroneStateVector(0)[0:3].copy()
        self.evader_pos = self._getDroneStateVector(1)[0:3].copy()
        return self._get_obs(), info

    def step(self, action):
        chaser_action = np.clip(action, -1.0, 1.0).astype(np.float32)
        evader_action = self._evader_action()
        full_action = np.vstack([chaser_action, evader_action])

        # 부모 클래스의 step()이 실제 물리(로터 추력/토크 -> 강체 시뮬레이션)를 전부 처리함
        _, _, _, _, info = super().step(full_action)
        self.step_count += 1

        chaser_state = self._getDroneStateVector(0)
        evader_state = self._getDroneStateVector(1)
        self.chaser_pos = chaser_state[0:3].copy()
        self.evader_pos = evader_state[0:3].copy()

        dist = np.linalg.norm(self.chaser_pos - self.evader_pos)
        chaser_oob = self._out_of_bounds(self.chaser_pos)
        evader_oob = self._out_of_bounds(self.evader_pos)

        terminated = False
        truncated = False
        reward = -0.1 * dist - 0.01
        info = {}

        if dist < self.catch_radius:
            reward += 50.0
            terminated = True
            info["result"] = "caught"
        elif chaser_oob:
            reward -= 20.0
            terminated = True
            info["result"] = "chaser_out_of_bounds"
        elif evader_oob:
            reward += 5.0
            terminated = True
            info["result"] = "evader_out_of_bounds"
        elif self.step_count >= self.max_steps:
            reward -= 5.0
            truncated = True
            info["result"] = "time_up_evader_wins"

        return self._get_obs(), reward, terminated, truncated, info

    def _get_obs(self):
        chaser_state = self._getDroneStateVector(0)
        evader_state = self._getDroneStateVector(1)
        chaser_pos = chaser_state[0:3]
        chaser_vel = chaser_state[10:13]
        evader_pos = evader_state[0:3]
        evader_vel = evader_state[10:13]
        rel_pos = evader_pos - chaser_pos
        rel_vel = evader_vel - chaser_vel
        return np.concatenate([chaser_pos, chaser_vel, rel_pos, rel_vel]).astype(np.float32)

    # BaseRLAviary가 내부적으로 요구하는 추상 메서드들 - 우리는 step()에서 직접
    # 보상/종료를 계산하므로 최소한의 형태로만 채워둠 (실제로 쓰이지 않음)
    def _computeReward(self):
        return 0.0

    def _computeTerminated(self):
        return False

    def _computeTruncated(self):
        return False

    def _computeInfo(self):
        return {}


if __name__ == "__main__":
    env = DroneTagPyBulletEnv(seed=0)
    for ep in range(2):
        obs, _ = env.reset()
        total_r = 0.0
        for t in range(env.max_steps):
            action = env.action_space.sample()
            obs, r, term, trunc, info = env.step(action)
            total_r += r
            if term or trunc:
                print(f"episode {ep}: steps={t+1}, reward={total_r:.2f}, {info}")
                break
    env.close()


## 3. 환경이 잘 도는지 빠르게 확인 (학습 없이 랜덤 액션 2판)
실제 물리 시뮬레이션이라 이전보다 한 스텝당 계산이 더 걸려. 여기서 에러가 나면 그 에러 메시지를 그대로 복사해서 알려줘 — 바로 고쳐줄게.

In [ ]:
from drone_tag_env import DroneTagPyBulletEnv as DroneTagEnv

env = DroneTagEnv(seed=0)
for ep in range(2):
    obs, _ = env.reset()
    total_r = 0.0
    for t in range(env.max_steps):
        action = env.action_space.sample()
        obs, r, term, trunc, info = env.step(action)
        total_r += r
        if term or trunc:
            print(f'episode {ep}: steps={t+1}, reward={total_r:.2f}, {info}')
            break
env.close()


## 4. 학습 시작 (술래 PPO 학습) — 반복 실행 가능
이 칸은 **여러 번 반복해서 눌러도 돼**. 처음 누르면 새로 시작하고, 그 다음부터는 저장된 모델을 자동으로 불러와서 이어서 학습해. 즉 `1번 누르기 → 결과 확인(5번) → 부족하면 4번 다시 누르기` 를 원하는 만큼 반복하면 돼.

Google Drive에 자동 백업되니까, 세션이 끊기거나 나중에 다시 들어와도 이어서 학습할 수 있어. 처음 실행하면 Drive 접근 권한 승인 팝업이 뜨는데, 승인만 누르면 됨.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.callbacks import EvalCallback

ADDITIONAL_TIMESTEPS = 100_000   # 이 칸을 누를 때마다 이만큼씩 추가로 학습됨. 필요하면 숫자만 바꿔도 됨
N_ENVS = 4   # 실제 물리엔진(PyBullet)은 병렬 환경 하나하나가 무거워서 8 -> 4로 줄임
OUT_PATH = 'models/chaser_phase1.zip'
LOGDIR = 'logs/phase1'
DRIVE_DIR = '/content/drive/MyDrive/drone_tag_rl'
DRIVE_MODEL_PATH = os.path.join(DRIVE_DIR, 'chaser_phase1.zip')

os.makedirs('models', exist_ok=True)
os.makedirs(LOGDIR, exist_ok=True)
os.makedirs(DRIVE_DIR, exist_ok=True)

# Drive에 백업본이 있는데 로컬(이번 세션)엔 없으면 먼저 복사해옴 (세션이 새로 시작된 경우)
if os.path.exists(DRIVE_MODEL_PATH) and not os.path.exists(OUT_PATH):
    shutil.copy(DRIVE_MODEL_PATH, OUT_PATH)
    print('Drive에서 이전 학습 모델을 불러왔어')

train_env = make_vec_env(lambda: DroneTagEnv(), n_envs=N_ENVS)
eval_env = make_vec_env(lambda: DroneTagEnv(), n_envs=1)

eval_callback = EvalCallback(
    eval_env,
    best_model_save_path=os.path.join(LOGDIR, 'best_model'),
    log_path=LOGDIR,
    eval_freq=max(10_000 // N_ENVS, 1),
    n_eval_episodes=10,
    deterministic=True,
)

if os.path.exists(OUT_PATH):
    print(f'기존 모델을 불러와서 이어서 학습함: {OUT_PATH}')
    model = PPO.load(OUT_PATH, env=train_env)
    print(f'지금까지 누적 학습 스텝: {model.num_timesteps:,}')
else:
    print('저장된 모델이 없어서 새로 시작함')
    model = PPO(
        'MlpPolicy',
        train_env,
        verbose=1,
        tensorboard_log=LOGDIR,
        n_steps=1024,
        batch_size=256,
        learning_rate=3e-4,
        gamma=0.99,
        gae_lambda=0.95,
        ent_coef=0.01,
    )

model.learn(total_timesteps=ADDITIONAL_TIMESTEPS, callback=eval_callback, reset_num_timesteps=False)
model.save(OUT_PATH)
shutil.copy(OUT_PATH, DRIVE_MODEL_PATH)  # Drive에도 백업

print(f'이번 실행 후 누적 학습 스텝: {model.num_timesteps:,}')
print(f'모델 저장 완료 (로컬 + Drive): {OUT_PATH}')
print('부족하면 이 칸을 다시 눌러서 이어서 학습시키면 돼')


## 5. 결과 확인 (궤적 애니메이션 HTML 생성 + 다운로드)
이 칸을 누르면 `results.html` 파일이 만들어지고, 태블릿으로 자동 다운로드됨. 다운받은 파일을 그냥 브라우저에서 열면 술래/도망자 움직임을 재생해서 볼 수 있어.
나중에 더 학습시키고 다시 이 칸만 눌러서 새로 뽑으면 결과가 갱신돼.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from stable_baselines3 import PPO

MODEL_PATH = 'models/chaser_phase1.zip'
EPISODES = 50

model = PPO.load(MODEL_PATH)
env = DroneTagEnv()

results = {'caught': 0, 'chaser_out_of_bounds': 0, 'evader_out_of_bounds': 0, 'time_up_evader_wins': 0}
episodes_data = []

for ep_idx in range(EPISODES):
    obs, _ = env.reset()
    traj_c, traj_e = [env.chaser_pos.copy()], [env.evader_pos.copy()]
    while True:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, term, trunc, info = env.step(action)
        traj_c.append(env.chaser_pos.copy())
        traj_e.append(env.evader_pos.copy())
        if term or trunc:
            results[info['result']] += 1
            break
    episodes_data.append({
        'index': ep_idx,
        'result': info['result'],
        'chaser': [p.tolist() for p in traj_c],
        'evader': [p.tolist() for p in traj_e],
    })

print('=== 결과 ===')
for k, v in results.items():
    print(f'{k}: {v}/{EPISODES} ({100*v/EPISODES:.1f}%)')
print(f"\ncatch율: {100*results['caught']/EPISODES:.1f}%")

data_json = json.dumps(episodes_data, ensure_ascii=False)
ARENA_XY = env.arena_xy
CATCH_R = env.catch_radius

html = f'''<!DOCTYPE html>
<html lang="ko">
<head>
<meta charset="UTF-8">
<title>드론 술래잡기 - 결과 뷰어</title>
<style>
  body {{ font-family: -apple-system, sans-serif; background: #12141c; color: #eee; margin: 0; padding: 20px; }}
  h1 {{ font-size: 18px; font-weight: 600; }}
  #canvas-wrap {{ display: flex; justify-content: center; margin-top: 16px; }}
  canvas {{ background: #1c1f2b; border-radius: 8px; }}
  .controls {{ display: flex; gap: 12px; align-items: center; justify-content: center; margin-top: 16px; flex-wrap: wrap; }}
  select, button {{ background: #2a2e3d; color: #eee; border: 1px solid #444; border-radius: 6px; padding: 6px 12px; font-size: 14px; }}
  button {{ cursor: pointer; }}
  button:hover {{ background: #3a3f52; }}
  .legend {{ display: flex; gap: 16px; justify-content: center; margin-top: 10px; font-size: 13px; color: #aaa; }}
  .dot {{ display: inline-block; width: 10px; height: 10px; border-radius: 50%; margin-right: 4px; vertical-align: middle; }}
  #result-badge {{ text-align: center; margin-top: 8px; font-size: 13px; color: #9aa; }}
</style>
</head>
<body>
  <h1>드론 술래잡기 결과 뷰어</h1>
  <div class="legend">
    <span><span class="dot" style="background:#4da3ff"></span>술래 (chaser)</span>
    <span><span class="dot" style="background:#ff5c7a"></span>도망자 (evader)</span>
  </div>
  <div id="canvas-wrap"><canvas id="c" width="600" height="600"></canvas></div>
  <div id="result-badge"></div>
  <div class="controls">
    <select id="epSelect"></select>
    <button id="playBtn">▶ 재생</button>
    <button id="resetBtn">⏮ 처음으로</button>
    <label>속도:
      <select id="speedSelect">
        <option value="1">1x</option>
        <option value="2" selected>2x</option>
        <option value="4">4x</option>
      </select>
    </label>
  </div>
<script>
const EPISODES = {data_json};
const ARENA_XY = {ARENA_XY};
const CATCH_R = {CATCH_R};
const canvas = document.getElementById('c');
const ctx = canvas.getContext('2d');
const W = canvas.width, H = canvas.height;
const scale = (W / 2) / (ARENA_XY * 1.1);
const cx = W / 2, cy = H / 2;
function toScreen(x, y) {{ return [cx + x * scale, cy - y * scale]; }}
const epSelect = document.getElementById('epSelect');
EPISODES.forEach((ep, i) => {{
  const opt = document.createElement('option');
  opt.value = i;
  opt.textContent = `에피소드 ${{ep.index}} — ${{ep.result}}`;
  epSelect.appendChild(opt);
}});
let currentEp = EPISODES[0];
let frame = 0;
let playing = false;
let speed = 2;
function drawArena() {{
  ctx.clearRect(0, 0, W, H);
  ctx.strokeStyle = '#3a3f52';
  ctx.lineWidth = 2;
  const [x0, y0] = toScreen(-ARENA_XY, ARENA_XY);
  const [x1, y1] = toScreen(ARENA_XY, -ARENA_XY);
  ctx.strokeRect(x0, y0, x1 - x0, y1 - y0);
}}
function drawTrail(traj, upTo, color) {{
  ctx.strokeStyle = color;
  ctx.globalAlpha = 0.5;
  ctx.lineWidth = 2;
  ctx.beginPath();
  for (let i = 0; i <= upTo; i++) {{
    const [sx, sy] = toScreen(traj[i][0], traj[i][1]);
    if (i === 0) ctx.moveTo(sx, sy); else ctx.lineTo(sx, sy);
  }}
  ctx.stroke();
  ctx.globalAlpha = 1.0;
}}
function drawDot(pos, color, radius) {{
  const [sx, sy] = toScreen(pos[0], pos[1]);
  ctx.fillStyle = color;
  ctx.beginPath();
  ctx.arc(sx, sy, radius, 0, Math.PI * 2);
  ctx.fill();
}}
function render() {{
  drawArena();
  const c = currentEp.chaser, e = currentEp.evader;
  const upTo = Math.min(frame, c.length - 1);
  drawTrail(c, upTo, '#4da3ff');
  drawTrail(e, upTo, '#ff5c7a');
  drawDot(c[upTo], '#4da3ff', 8);
  drawDot(e[upTo], '#ff5c7a', 8);
  const [sx, sy] = toScreen(c[upTo][0], c[upTo][1]);
  ctx.strokeStyle = 'rgba(77,163,255,0.3)';
  ctx.beginPath();
  ctx.arc(sx, sy, CATCH_R * scale, 0, Math.PI * 2);
  ctx.stroke();
  document.getElementById('result-badge').textContent =
    `스텝 ${{upTo}} / ${{c.length - 1}} — 결과: ${{currentEp.result}}`;
}}
function tick() {{
  if (!playing) return;
  frame += speed;
  if (frame >= currentEp.chaser.length - 1) {{
    frame = currentEp.chaser.length - 1;
    playing = false;
    document.getElementById('playBtn').textContent = '▶ 재생';
  }}
  render();
  if (playing) requestAnimationFrame(tick);
}}
document.getElementById('playBtn').onclick = () => {{
  playing = !playing;
  document.getElementById('playBtn').textContent = playing ? '⏸ 일시정지' : '▶ 재생';
  if (playing) requestAnimationFrame(tick);
}};
document.getElementById('resetBtn').onclick = () => {{ frame = 0; render(); }};
document.getElementById('speedSelect').onchange = (e) => {{ speed = Number(e.target.value); }};
epSelect.onchange = (e) => {{
  currentEp = EPISODES[Number(e.target.value)];
  frame = 0;
  playing = false;
  document.getElementById('playBtn').textContent = '▶ 재생';
  render();
}};
render();
</script>
</body>
</html>
'''

with open('results.html', 'w', encoding='utf-8') as f:
    f.write(html)

print('results.html 생성 완료')

from google.colab import files
files.download('results.html')


## GitHub Pages로 실시간 결과 페이지 만들기

**최초 1회만 설정**
1. 레포 → `Settings` → `Pages`
2. `Build and deployment` → `Source`: `Deploy from a branch`
3. `Branch`: `main`, 폴더: `/docs` 선택 → `Save`
4. 몇 분 후 `https://<username>.github.io/<repo>/` 로 접속하면 페이지 보임

**그 다음부터는**
- 아래 칸에 본인 레포 주소랑 토큰만 채워서 실행하면, `results.html`이 자동으로 `docs/index.html`로 레포에 올라감
- 5번 칸(평가) → 이 칸 실행 반복하면, 매번 그 URL 새로고침만 해도 최신 결과가 보임 (다운로드 필요 없음)
- 토큰은 GitHub `Settings` → `Developer settings` → `Personal access tokens`에서 발급 (repo 권한 체크)

In [ ]:
# 아래 두 줄만 본인 정보로 채우고 실행하면 결과 페이지가 자동으로 GitHub Pages에 반영됨
REPO_URL = 'https://github.com/<username>/<repo>.git'   # 본인 레포 주소로 변경
GITHUB_TOKEN = ''  # GitHub Settings > Developer settings > Personal access tokens 에서 발급 (repo 권한)

import os, shutil

if GITHUB_TOKEN:
    auth_url = REPO_URL.replace('https://', f'https://{GITHUB_TOKEN}@')
    if not os.path.exists('repo_push'):
        !git clone {auth_url} repo_push
    else:
        !cd repo_push && git pull

    os.makedirs('repo_push/docs', exist_ok=True)
    shutil.copy('results.html', 'repo_push/docs/index.html')
    if os.path.exists('models'):
        os.makedirs('repo_push/models', exist_ok=True)
        for fname in os.listdir('models'):
            shutil.copy(os.path.join('models', fname), os.path.join('repo_push/models', fname))

    !cd repo_push && git add . && git commit -m "colab 결과 갱신" && git push
    print('푸시 완료 - 몇 분 뒤 GitHub Pages URL 새로고침하면 최신 결과 보임')
else:
    print('GITHUB_TOKEN을 채운 다음 다시 실행해줘')
